# BE Optimisation — Reactive Covering

Planification de la réservation de passes satellites pour couvrir un ensemble de requêtes d'observation au moindre coût.

**Sommaire**
1. Données et outils d'affichage
2. Heuristique gloutonne
3. Modèle CPLEX — passes complètes
4. Modèle CPLEX — passes partielles
5. Visualisation des réservations
6. Animation de la couverture
7. Front de Pareto coût / satisfaction
8. Comparaison des coûts

## 1. Données et outils d'affichage

Chargement de l'instance (accès, requêtes, passes, coûts) et définition de la fonction de tracé des solutions.

In [ ]:
import json
import math
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from docplex.mp.model import Model
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.lines import Line2D
from IPython.display import Image, display



with open('data/data2.json') as f:  
    data = json.load(f)

accesses = data['Accesses']
obsRequests = data['ObservationRequests']
nObsRequests = len(obsRequests)
satPasses = data['Passes']
c_unit = data["fixedPassCost"]
c_temps = data["passCostPerTimeUnit"]
minVisiDuration = data['minVisiDuration']



def plotSolution(data,solution):

    # Charger le GeoJSON
    with open("countries.geo.json") as f:
        dataCountries = json.load(f)

    # get some data
    minVisiDuration = data['minVisiDuration']
    AOIs = data['AOIs']
    accesses = data['Accesses']
    satPasses = data['Passes']
    
    # get the booking strategy
    bookings = solution['Bookings']
    bookings.sort(key=lambda item: satPasses[item["passId"]]['startDate'])    
    print("Bookings:")
    for booking in bookings:
        print(booking)
    
    # show the passes used
    ncols = 3
    nrows = math.ceil(len(solution['Bookings']) / ncols)
    fig, axs = plt.subplots(nrows=nrows,ncols=ncols, figsize=(30,30))
    axs = axs.flatten()

    i = 0
    for booking in solution['Bookings']:
        passId = booking["passId"]
        satPass = satPasses[passId]
        ax = axs[i]
        
        # Dessiner les frontières
        for feature in dataCountries["features"]:
            geometry = feature["geometry"]
    
            if geometry["type"] == "Polygon":
                for coords in geometry["coordinates"]:
                    xs, ys = zip(*coords)
                    ax.fill(xs, ys, facecolor="lightgray", edgecolor="black", linewidth=0.5)

            elif geometry["type"] == "MultiPolygon":
                for polygon in geometry["coordinates"]:
                    for coords in polygon:
                        xs, ys = zip(*coords)
                        ax.fill(xs, ys, facecolor="lightgray", edgecolor="black", linewidth=0.5)

        
        ax.set_xlim(-9, 13)
        ax.set_ylim(38, 56)
        ax.set_title("Pass #{} [{},{}]".format(passId,satPass['startDate'],satPass['endDate']))
        ax.plot([aoi['lon'] for aoi in AOIs], [aoi['lat'] for aoi in AOIs], 'kx')

        # show the reservations
        xpoints = []
        ypoints = []
        colors = []
        for accessId in satPass['accessIds']:
            access = accesses[accessId]
            aoi = AOIs[access['aoiId']]
            xpoints.append(aoi['lon'])
            ypoints.append(aoi['lat'])
            if (booking['bookingStart'] <= access['end'] - minVisiDuration) and (access['start'] + minVisiDuration <= booking['bookingEnd']):
                colors.append('r')
            else:
                colors.append('b')       
        ax.scatter(xpoints,ypoints,marker='o',c=colors)
        
        i = i + 1


plt.show()



## 2. Heuristique gloutonne

Solution de référence : les passes sont triées par nombre d'accès par unité de temps puis sélectionnées tant qu'elles couvrent de nouvelles requêtes.

In [ ]:
import random
import json

# GLOUTON

satisfied = [False for i in range(nObsRequests)]
nObsRequestsToCover = nObsRequests
nPasses = len(satPasses)
remainingSatPassesIds = list(range(nPasses))

#random.shuffle(remainingSatPassesIds)
alpha = 0
#remainingSatPassesIds.sort(key=lambda index:alpha*(satPasses[index]['end']-satPasses[index]['start'])+(1-alpha)*len(satPasses[index]['accessIds']), reverse=True)

# Tri statique par ratio : Rentabilité = (Nombre d'accès potentiels) / (Durée de la passe)
# Plus le score est haut, plus la passe est intéressante (beaucoup d'accès pour un temps court)
remainingSatPassesIds.sort(
    key=lambda idx: len(satPasses[idx]['accessIds']) / (satPasses[idx]['end'] - satPasses[idx]['start']), 
    reverse=True
)

selectedPassIds = []
for i in remainingSatPassesIds:
    # try to select the ith path
    satPass = satPasses[i]
    keepPass = False
    for accessId in satPass['accessIds']:
        access = accesses[accessId]
        for j in range(nObsRequests):
            obsRequest = obsRequests[j]
            if not satisfied[j] and accessId in obsRequest['accessIds']:
                satisfied[j] = True
                keepPass = True
                nObsRequestsToCover -= 1
                
    if keepPass:
        selectedPassIds.append(i)
            
    if nObsRequestsToCover == 0:
        break


print("selectedPasses:", selectedPassIds)
print("nSelectedPasses:", len(selectedPassIds))

c_unit=120
c_temps=1
c_total=0

scr=satisfied.count(True)/nObsRequests

# export the solution to json
bookings = []
for i in selectedPassIds:
    satPass = satPasses[i]
    bookingStart = (satPass['start'] + 60)
    bookingEnd = (satPass['end'] - 60)
    bookings.append({"passId": i, "passStart": satPass['start'], "passEnd": satPass['end'], "bookingStart": bookingStart, "bookingEnd": bookingEnd})
    c_total=c_total+c_unit+c_temps*(bookingEnd -bookingStart)
print("Cout total",c_total)
print("Score de Satisfaction", scr)
jsonString = json.dumps({"Bookings": bookings})
jsonFile = open("mySolution.json", "w")
jsonFile.write(jsonString)
jsonFile.close()




## 3. Modèle CPLEX — passes complètes

Programme linéaire en nombres entiers où chaque passe est réservée sur toute sa durée, suivi de l'analyse de la solution obtenue.

In [ ]:
#CPLEX passages complets

from docplex.mp.model import Model
import numpy as np
import random
import json

# import the data
dataFile = open('data/data2.json')  
data = json.load(dataFile)
accesses = data['Accesses']
obsRequests = data['ObservationRequests'];
nObsRequests = len(obsRequests)
satPasses = data['Passes'];

def request_to_passIDs(requestID):
    pass_IDs=[]
    for acessID in obsRequests[requestID]["accessIds"]:
        pass_IDs.append(accesses[acessID]["passId"])
    return pass_IDs

def request_to_accessIDs(requestID):
    access_IDs=obsRequests[requestID]["accessIds"]
    return access_IDs

c_unit=data["fixedPassCost"]
c_temps=data["passCostPerTimeUnit"]

# Creation d'un modèle
model = Model(name='Reactive Covering')


#VARIABLES

#requetes satisfaites
r = model.binary_var_list(range(nObsRequests), name='r')
#passes reservees
p = model.binary_var_list(range(len(satPasses)), name='p')
#CONTRAINTES

#Contrainte au moins un passage réservé si requete satisfaite
for i in range(nObsRequests):
    passIDs_i = request_to_passIDs(i)
    model.add_constraint(r[i]<=model.sum(p[j] for j in passIDs_i))
                                    

                                    
# fonction objectif
cout_max = sum(c_unit + c_temps * (sp['end'] - sp['start']) for sp in satPasses)
alpha = 0.5
satisfaction = model.sum(r[i] for i in range(nObsRequests)) / nObsRequests
cout = model.sum(p[j]*(c_unit + c_temps*(satPasses[j]["end"] - satPasses[j]["start"])) for j in range(len(satPasses)))
model.maximize(alpha * satisfaction - (1 - alpha) * cout / cout_max)
model.set_time_limit(120)
model.print_information()

# Résolution du problème
solution = model.solve(log_output=True)

# --- SAUVEGARDE JSON
bookings_complet = []
for i in range(len(satPasses)):
    if solution.get_value(p[i]) > 0.5:
        sp = satPasses[i]
        bookings_complet.append({
            "passId": i, 
            "passStart": sp['start'], 
            "passEnd": sp['end'], 
            "bookingStart": sp['start'], 
            "bookingEnd": sp['end']
        })
        
with open("mySolution_complet.json", "w") as f:
    json.dump({"Bookings": bookings_complet}, f)
print("\nFichier 'mySolution_complet.json' généré avec succès !")



In [ ]:
# Affichage de la solution et des métriques clés
if solution is None:
    print("NO solution")
else:
    print(f"Statut : {model.solve_details.status} (gap {model.solve_details.mip_relative_gap:.2%})")
    
    # --- 1. Taux de satisfaction global ---
    nb_satisfaites = sum(1 for j in range(nObsRequests) if solution.get_value(r[j]) > 0.5)
    taux_satisfaction = (nb_satisfaites / nObsRequests) * 100
    print(f"\n--- SATISFACTION ---")
    print(f"Requêtes satisfaites : {nb_satisfaites} / {nObsRequests} ({taux_satisfaction:.2f}%)")

    # --- 2. Taux de satisfaction par priorité ---
    # Initialisation des compteurs
    req_par_prio = {1: {'total': 0, 'satisfaites': 0}, 
                    2: {'total': 0, 'satisfaites': 0}, 
                    3: {'total': 0, 'satisfaites': 0}}
    
    for j in range(nObsRequests):
        prio = obsRequests[j]['priority']
        req_par_prio[prio]['total'] += 1
        if solution.get_value(r[j]) > 0.5:
            req_par_prio[prio]['satisfaites'] += 1

    print("\nDétail par priorité :")
    for prio in [1, 2, 3]:
        total = req_par_prio[prio]['total']
        satisfaites = req_par_prio[prio]['satisfaites']
        taux = (satisfaites / total * 100) if total > 0 else 0
        print(f"Priorité {prio} (P{prio}) : {satisfaites} / {total} couvertes ({taux:.2f}%)")

    # --- 3. Coût total détaillé ---
    cout_fixe_total = 0
    cout_variable_total = 0
    passes_choisies = 0
    
    for i in range(len(satPasses)):
        if solution.get_value(p[i]) > 0.5:
            passes_choisies += 1
            cout_fixe_total += c_unit
            
        
            duree = satPasses[i]['end'] - satPasses[i]['start']
            cout_variable_total += c_temps * duree

    cout_total = cout_fixe_total + cout_variable_total
    print(f"\n--- COÛTS ---")
    print(f"Passes réservées : {passes_choisies} / {len(satPasses)}")
    print(f"Coût fixe total : {cout_fixe_total:.2f}")
    print(f"Coût variable total : {cout_variable_total:.2f}")
    print(f"Coût total de réservation : {cout_total:.2f}")

## 4. Modèle CPLEX — passes partielles

Extension du modèle précédent : la fenêtre de réservation de chaque passe est ajustée pour ne payer que le temps utile.

In [ ]:
#CPLEX passages partiels

from docplex.mp.model import Model
import numpy as np
import random
import json

# import the data
dataFile = open('data/data2.json')  
data = json.load(dataFile)
accesses = data['Accesses']
obsRequests = data['ObservationRequests'];
nObsRequests = len(obsRequests)
satPasses = data['Passes'];

def request_to_passIDs(requestID):
    pass_IDs=[]
    for acessID in obsRequests[requestID]["accessIds"]:
        pass_IDs.append(accesses[acessID]["passId"])
    return pass_IDs

def request_to_accessIDs(requestID):
    access_IDs=obsRequests[requestID]["accessIds"]
    return access_IDs

c_unit=data["fixedPassCost"]
c_temps=data["passCostPerTimeUnit"]

# Creation d'un modèle
model = Model(name='Reactive Covering')


#VARIABLES

#requetes satisfaites
r = model.binary_var_list(range(nObsRequests), name='r')
#passes reservees
p = model.binary_var_list(range(len(satPasses)), name='p')
#accès utilisés 
a = model.binary_var_list(range(len(accesses)), name='a')
#t_début
t_debut = model.continuous_var_list(range(len(satPasses)),lb=min(satPasses[i]['start'] for i in range(len(satPasses))), ub=max(satPasses[i]['end'] for i in range(len(satPasses))), name='t_debut')
#t_fin
t_fin = model.continuous_var_list(range(len(satPasses)),lb=min(satPasses[i]['start'] for i in range(len(satPasses))), ub=max(satPasses[i]['end'] for i in range(len(satPasses))), name='t_fin')

#CONTRAINTES

#Contrainte au moins un passage réservé si requete satisfaite
for i in range(nObsRequests):
    passIDs_i = request_to_passIDs(i)
    model.add_constraint(r[i]<=model.sum(p[j] for j in passIDs_i))
                                    
#Si l'acces est utilisé, sa passe associée doit être réservée    
for i in range(len(accesses)):
    model.add_constraint(a[i]<=p[accesses[i]["passId"]])
                                    
#Au moins un accès de la requete ri est utilisé si requete satisfaite
for i in range(nObsRequests):
    access_IDs_i = request_to_accessIDs(i)
    model.add_constraint(r[i]<=model.sum(a[j] for j in access_IDs_i))
            
#Contraintes sur t_debut & t_fin
for i in range(len(satPasses)):
    model.add_constraint(t_debut[i]<=t_fin[i])
    model.add_constraint(t_debut[i]>=satPasses[i]['start'])
    model.add_constraint(t_fin[i]<=satPasses[i]['end'])
    
#Big M pour la contrainte de t_debut et t_fin sur les accès d'une passes
for i in range(len(accesses)):
    j = accesses[i]["passId"]
    M = satPasses[j]["end"] - satPasses[j]["start"] + 30
    model.add_constraint(t_debut[j]<=accesses[i]["end"]-30 + M*(1-a[i]))
    model.add_constraint(t_fin[j]>=accesses[i]["start"]+30 - M*(1-a[i]))
                                    
                                    
                                    
# fonction objectif
cout_max = sum(c_unit + c_temps * (sp['end'] - sp['start']) for sp in satPasses)
alpha = 0.5
satisfaction = model.sum(r[i] for i in range(nObsRequests)) / nObsRequests
cout = model.sum(p[j]*(c_unit + c_temps*(t_fin[j] - t_debut[j])) for j in range(len(satPasses)))
model.maximize(alpha * satisfaction - (1 - alpha) * cout / cout_max)
#model.set_time_limit(120)
model.set_time_limit(120)
model.print_information()

# Résolution du problème
solution = model.solve(log_output=True)

# --- SAUVEGARDE JSON
bookings_partiel = []
for i in range(len(satPasses)):
    if solution.get_value(p[i]) > 0.5:
        sp = satPasses[i]
        bookings_partiel.append({
            "passId": i, 
            "passStart": sp['start'], 
            "passEnd": sp['end'], 
            "bookingStart": solution.get_value(t_debut[i]), 
            "bookingEnd": solution.get_value(t_fin[i])
        })
        
with open("mySolution_partiel.json", "w") as f:
    json.dump({"Bookings": bookings_partiel}, f)
print("\nFichier 'mySolution_partiel.json' généré avec succès !")



In [ ]:
# Affichage de la solution et des métriques clés
if solution is None:
    print("NO solution")
else:
    print(f"Statut : {model.solve_details.status} (gap {model.solve_details.mip_relative_gap:.2%})")
    
    # --- 1. Taux de satisfaction global ---
    nb_satisfaites = sum(1 for j in range(nObsRequests) if solution.get_value(r[j]) > 0.5)
    taux_satisfaction = (nb_satisfaites / nObsRequests) * 100
    print(f"\n--- SATISFACTION ---")
    print(f"Requêtes satisfaites : {nb_satisfaites} / {nObsRequests} ({taux_satisfaction:.2f}%)")

    # --- 2. Taux de satisfaction par priorité ---
    # Initialisation des compteurs
    req_par_prio = {1: {'total': 0, 'satisfaites': 0}, 
                    2: {'total': 0, 'satisfaites': 0}, 
                    3: {'total': 0, 'satisfaites': 0}}
    
    for j in range(nObsRequests):
        prio = obsRequests[j]['priority']
        req_par_prio[prio]['total'] += 1
        if solution.get_value(r[j]) > 0.5:
            req_par_prio[prio]['satisfaites'] += 1

    print("\nDétail par priorité :")
    for prio in [1, 2, 3]:
        total = req_par_prio[prio]['total']
        satisfaites = req_par_prio[prio]['satisfaites']
        taux = (satisfaites / total * 100) if total > 0 else 0
        print(f"Priorité {prio} (P{prio}) : {satisfaites} / {total} couvertes ({taux:.2f}%)")

    # --- 3. Coût total détaillé ---
    cout_fixe_total = 0
    cout_variable_total = 0
    passes_choisies = 0
    
    for i in range(len(satPasses)):
        if solution.get_value(p[i]) > 0.5:
            passes_choisies += 1
            cout_fixe_total += c_unit
            
            # ATTENTION : Adaptez cette ligne selon le modèle (Complets ou Partiels)
            # Pour Modèle A (Complets) : duree = satPasses[i]['end'] - satPasses[i]['start']
            # Pour Modèle B (Partiels) : duree = solution.get_value(t_fin[i]) - solution.get_value(t_debut[i])
            duree = solution.get_value(t_fin[i]) - solution.get_value(t_debut[i]) # Version Partiels
            cout_variable_total += c_temps * duree

    cout_total = cout_fixe_total + cout_variable_total
    print(f"\n--- COÛTS ---")
    print(f"Passes réservées : {passes_choisies} / {len(satPasses)}")
    print(f"Coût fixe total : {cout_fixe_total:.2f}")
    print(f"Coût variable total : {cout_variable_total:.2f}")
    print(f"Coût total de réservation : {cout_total:.2f}")

## 5. Visualisation des réservations

Chronogramme des passes réservées, de leurs fenêtres de réservation et des accès effectivement couverts.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Chargement du JSON
solution = json.load(open('mySolution_partiel.json'))

if solution is not None and 'Bookings' in solution:
    
    bookings = solution['Bookings']
    # 1. Récupérer les IDs des passes directement depuis la liste des réservations JSON
    passes_choisies = [b['passId'] for b in bookings]
    min_visi = data['minVisiDuration']
    
    # Création de la figure
    fig, ax = plt.subplots(figsize=(12, max(5, len(passes_choisies) * 0.7)))
    
    for y_pos, b in enumerate(bookings):
        pass_idx = b['passId']
        sp = satPasses[pass_idx]
        p_start = sp['start']
        p_end = sp['end']
        duration = p_end - p_start
        
        # Temps de réservation extraits du JSON (ramenés à t=0 pour l'affichage)
        t_d = b['bookingStart'] - p_start
        t_f = b['bookingEnd'] - p_start
        
        # A. DESSIN DE LA PASSE COMPLÈTE (Fond vert sombre)
        ax.barh(y_pos, duration, left=0, height=0.6, color='#3a5a40', alpha=0.8, 
                label='Fenêtre de visibilité du satellite' if y_pos == 0 else "")
        
        # B. DESSIN DE LA RÉSERVATION (Fenêtre rouge pointillée)
        ax.barh(y_pos, t_f - t_d, left=t_d, height=0.65, color='#a12828', alpha=0.5, 
                edgecolor='red', linestyle='--', linewidth=2, hatch='///',
                label='Temps d\'allumage réservé' if y_pos == 0 else "")
        
        # C. DESSIN DES ACCÈS (On teste manuellement si l'accès est couvert par la réservation)
        acces_utilises = []
        for acc_id in sp['accessIds']:
            acc = accesses[acc_id]
            # Test métier : la réservation recouvre au moins 30s de cet accès
            if b['bookingStart'] <= acc['end'] - min_visi and acc['start'] + min_visi <= b['bookingEnd']:
                acces_utilises.append(acc)
        
        for i_acc, acc in enumerate(acces_utilises):
            a_start = acc['start'] - p_start
            a_end = acc['end'] - p_start
            
            # Rectangle de l'accès (en orange)
            ax.barh(y_pos, a_end - a_start, left=a_start, height=0.3, color='#e28743', 
                    edgecolor='black', linewidth=1, zorder=3, 
                    label='Accès utilisé (Requête)' if y_pos == 0 and i_acc == 0 else "")
            
            # D. MARQUEUR DES 30 SECONDES (La contrainte métier)
            overlap_start = max(a_start, t_d)
            ax.plot([overlap_start, overlap_start + min_visi], [y_pos - 0.25, y_pos - 0.25], 
                    color='red', linewidth=3, zorder=4, solid_capstyle='butt',
                    label='Contrainte Min Visi (30s)' if y_pos == 0 and i_acc == 0 else "")

    # Mise en forme esthétique du graphique
    ax.set_yticks(range(len(passes_choisies)))
    ax.set_yticklabels([f"Passe n°{i}" for i in passes_choisies], fontweight='bold')
    
    ax.set_xlabel("Temps (secondes écoulées depuis le début de la passe)", fontsize=11, fontweight='bold')
    ax.set_title("Timeline des Réservations et Couverture des Accès", fontsize=14, pad=15)
    
    # Nettoyage de la légende
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=10)
    
    plt.grid(axis='x', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()


## 6. Animation de la couverture

Animation de la progression de la couverture des zones d'intérêt au fil des passes réservées, exportée en GIF.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

# --- 1. Chargement des données ---
data = json.load(open('data/data2.json'))
solution = json.load(open('mySolution_complet.json'))

AOIs = data['AOIs']
accesses = data['Accesses']
obsRequests = data['ObservationRequests']
satPasses = data['Passes']
d = data['minVisiDuration']
nAOIs, nObsRequests = len(AOIs), len(obsRequests)

# Filtrer les réservations < 30s
vraies_reservations = [b for b in solution['Bookings'] if (b['bookingEnd'] - b['bookingStart']) >= d]
bookings = sorted(solution['Bookings'], key=lambda b: satPasses[b['passId']]['start'])
total_passes_reservees = len(bookings)

# Prétraitement
req_par_aoi = [[] for _ in range(nAOIs)]
total_par_prio = {1: 0, 2: 0, 3: 0}
for i, req in enumerate(obsRequests):
    req_par_aoi[req['aoiId']].append(i)
    total_par_prio[req['priority']] += 1

acces_req = [set(req['accessIds']) for req in obsRequests]

def couvert(acc, b):
    return b['bookingStart'] <= acc['end'] - d and acc['start'] + d <= b['bookingEnd']

# --- 2. Création de la figure (Rectangle vertical avec carte carrée) ---
# Un format 8x10.5 offre un beau rectangle vertical pour s'insérer sur un côté de slide
fig, (ax_map, ax_bar) = plt.subplots(2, 1, figsize=(8, 10.5), gridspec_kw={'height_ratios': [1, 0.4]})
fig.subplots_adjust(top=0.92, bottom=0.08, left=0.05, right=0.95, hspace=0.1)

fig.suptitle("Évolution temporelle des observations", fontsize=18, fontweight='bold')

# --- SETUP CARTE (Haut) ---
with open("countries.geo.json") as f:
    dataCountries = json.load(f)
for feature in dataCountries["features"]:
    geom = feature["geometry"]
    polys = [geom["coordinates"]] if geom["type"] == "Polygon" else geom["coordinates"]
    for poly in polys:
        for coords in poly:
            xs, ys = zip(*coords)
            ax_map.fill(xs, ys, facecolor="#e8e8e8", edgecolor="black", linewidth=0.5)

ax_map.set_xlim(-9, 13)
ax_map.set_ylim(38, 56)
ax_map.set_aspect('equal', adjustable='box') # Force la carte à garder des proportions réelles/carrées
ax_map.set_xticks([])
ax_map.set_yticks([])

lon = np.array([aoi['lon'] for aoi in AOIs])
lat = np.array([aoi['lat'] for aoi in AOIs])
cmap = plt.get_cmap('RdYlGn') 
scatter_aois = ax_map.scatter(lon, lat, s=80, zorder=3, edgecolors='black', linewidths=0.5)
cercles = ax_map.scatter([], [], s=300, facecolors='none', edgecolors='blue', linewidths=2.5, zorder=4)

# Compteur de passes incrusté
txt_passes = ax_map.text(0.97, 0.95, f"Passes : 0 / {total_passes_reservees}", 
                         transform=ax_map.transAxes, ha='right', va='top', 
                         fontsize=13, fontweight='bold', color='black',
                         bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5', alpha=0.9))

titre_map = ax_map.set_title("Initialisation", fontsize=14, pad=10)

# --- SETUP BARRES (Bas) ---
# Nouvelles priorités : P1 = Faible (12h) -> Vert, P3 = Urgente (2h) -> Rouge
prio_labels = ['Priorité 1 (12h)', 'Priorité 2 (4h)', 'Priorité 3 (2h)']
prio_colors = ['#2ca02c', '#ff7f0e', '#d62728']
bars = ax_bar.bar(prio_labels, [0, 0, 0], color=prio_colors, edgecolor='black', width=0.5)

ax_bar.set_ylim(0, 105)
ax_bar.set_ylabel("Complétion (%)", fontweight='bold', fontsize=12)
ax_bar.set_title("Couverture globale par niveau de priorité", fontsize=14)
ax_bar.tick_params(axis='x', labelsize=12)

bar_texts = [ax_bar.text(i, 5, "0%", ha='center', color='white', fontweight='bold', fontsize=13) for i in range(3)]

# --- 3. Simulation des étapes ---
satisfaite = [False] * nObsRequests
sat_par_prio = {1: 0, 2: 0, 3: 0}

images_data = [{
    "sat": list(satisfaite),
    "prio": dict(sat_par_prio),
    "vues": [],
    "passes_count": 0,
    "titre": "En attente du premier survol..."
}]

for step, b in enumerate(bookings):
    sp = satPasses[b['passId']]
    acces_couverts = {a for a in sp['accessIds'] if couvert(accesses[a], b)}
    aoi_vues = sorted({accesses[a]['aoiId'] for a in acces_couverts})

    nouvelles = 0
    for i in range(nObsRequests):
        if not satisfaite[i] and acces_req[i] & acces_couverts:
            satisfaite[i] = True
            sat_par_prio[obsRequests[i]['priority']] += 1
            nouvelles += 1

    images_data.append({
        "sat": list(satisfaite),
        "prio": dict(sat_par_prio),
        "vues": aoi_vues,
        "passes_count": step + 1,
        "titre": f"Observation validée | + {nouvelles} requêtes satisfaites"
    })

# Répéter la dernière frame
images_data += [images_data[-1]] * 4

# --- 4. Fonction de mise à jour ---
def update(frame):
    data_frame = images_data[frame]
    
    ratios_aoi = []
    sizes = []
    for k in range(nAOIs):
        if len(req_par_aoi[k]) == 0:
            ratios_aoi.append(1.0)
            sizes.append(20)
        else:
            sat_count = sum(data_frame["sat"][i] for i in req_par_aoi[k])
            ratio = sat_count / len(req_par_aoi[k])
            ratios_aoi.append(ratio)
            sizes.append(100 - (70 * ratio))
            
    scatter_aois.set_color(cmap(ratios_aoi))
    scatter_aois.set_sizes(sizes)
    scatter_aois.set_edgecolor('black')
    
    vues = data_frame["vues"]
    cercles.set_offsets(np.c_[lon[vues], lat[vues]] if vues else np.empty((0, 2)))
    titre_map.set_text(data_frame["titre"])
    
    for idx, p in enumerate([1, 2, 3]):
        tot = total_par_prio[p]
        taux = (data_frame["prio"][p] / tot * 100) if tot > 0 else 0
        bars[idx].set_height(taux)
        bar_texts[idx].set_text(f"{taux:.1f}%")
        bar_texts[idx].set_y(max(6, taux - 8) if taux > 15 else 5)
        
    txt_passes.set_text(f"Passes : {data_frame['passes_count']} / {total_passes_reservees}")
        
    return scatter_aois, cercles, titre_map, txt_passes

# --- 5. Exportation ---
anim = FuncAnimation(fig, update, frames=len(images_data), interval=800)
anim.save("dashboard_slide.gif", writer=PillowWriter(fps=1.2))
plt.close(fig)
display(Image(filename="dashboard_slide.gif"))


## 7. Front de Pareto coût / satisfaction

Résolution du modèle pour une série de budgets maximaux afin de tracer le compromis entre coût et taux de satisfaction.

In [ ]:
import matplotlib.pyplot as plt
from docplex.mp.model import Model
import math

# On prépare les listes pour le graphique
pareto_couts = []
pareto_satisfactions = []

# On fait varier le budget max alloué à la constellation (de 1000€ à 15000€)
budgets = list(range(1000, 16000, 1000))
min_visi = data['minVisiDuration'] # 30s

for budget_max in budgets:
    # 1. On crée un NOUVEAU modèle vierge pour ce budget
    m = Model(name=f'Pareto_{budget_max}')
    
    # 2. Déclaration des variables
    r = m.binary_var_list(nObsRequests, name='r')
    p = m.binary_var_list(len(satPasses), name='p')
    a = m.binary_var_list(len(accesses), name='a')
    t_debut = m.continuous_var_list(len(satPasses), lb=0, name='t_debut')
    t_fin = m.continuous_var_list(len(satPasses), lb=0, name='t_fin')

    # 3. On réinjecte TOUTES les contraintes
    # - Requêtes couvertes
    for j in range(nObsRequests):
        m.add_constraint(r[j] <= m.sum(a[k] for k in obsRequests[j]["accessIds"]))
        
    # - Bornes du temps des passes
    for i in range(len(satPasses)):
        m.add_constraint(t_debut[i] >= satPasses[i]['start'] * p[i])
        m.add_constraint(t_fin[i] <= satPasses[i]['end'] * p[i])
        m.add_constraint(t_fin[i] >= t_debut[i])
        
    # - Big-M pour le collage des temps d'accès
    for k in range(len(accesses)):
        pid = accesses[k]["passId"]
        big_M = satPasses[pid]["end"]
        m.add_constraint(t_debut[pid] <= (accesses[k]["end"] - min_visi) * a[k] + big_M * (1 - a[k]))
        m.add_constraint(t_fin[pid] >= (accesses[k]["start"] + min_visi) * a[k])
        m.add_constraint(a[k] <= p[pid])

    # 4. Fonction de Coût et Contrainte de Budget
    cout_total = m.sum(c_unit * p[i] + c_temps * (t_fin[i] - t_debut[i]) for i in range(len(satPasses)))
    m.add_constraint(cout_total <= budget_max)

    # 5. Nouvel Objectif : Maximiser les requêtes en respectant les priorités
    poids_prio = {1: 1000, 2: 100, 3: 10}
    gain_objectif = m.sum(r[j] * poids_prio[obsRequests[j]['priority']] for j in range(nObsRequests))
    m.maximize(gain_objectif)
    
    # 6. Résolution très rapide (30s par point suffisent pour le Pareto)
    m.set_time_limit(30)
    sol = m.solve(log_output=False)
    
    if sol:
        # Récupération de la satisfaction (en %)
        nb_sat = sum(sol.get_value(r[j]) for j in range(nObsRequests))
        taux = (nb_sat / nObsRequests) * 100
        # Récupération du coût réellement dépensé
        cout_reel = sol.get_value(cout_total)
        
        pareto_couts.append(cout_reel)
        pareto_satisfactions.append(taux)
        print(f"Budget max {budget_max}€ -> Coût réel: {cout_reel:.2f}€ | Satisfaction: {taux:.2f}%")
    else:
        print(f"Budget max {budget_max}€ -> Insoluble (budget trop faible)")

# 7. Affichage du vrai Front de Pareto
plt.figure(figsize=(10, 6))
plt.plot(pareto_couts, pareto_satisfactions, marker='o', linestyle='-', color='#1f77b4', linewidth=2, markersize=8)
plt.title("Front de Pareto : Taux de Satisfaction vs Coût", fontsize=14, pad=15)
plt.xlabel("Coût total de réservation (€)", fontsize=12)
plt.ylabel("Taux de satisfaction global (%)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.fill_between(pareto_couts, pareto_satisfactions, alpha=0.1, color='#1f77b4')
plt.show()

## 8. Comparaison des coûts

Décomposition en coûts fixes et variables des deux modèles CPLEX.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Données fictives à remplacer par les résultats réels de vos deux exécutions
modeles = ['Passages Complets (Étape 2)', 'Passages Partiels (Étape 3)']
couts_fixes = np.array([2400, 2400])    # Exemple : 20 passes réservées dans les deux cas (20 * 120)
couts_variables = np.array([8500, 3200]) # Exemple : réduction massive du temps d'allumage

plt.figure(figsize=(8, 6))
bars1 = plt.bar(modeles, couts_fixes, color='#2ca02c', label='Coût Fixe (Accès)')
bars2 = plt.bar(modeles, couts_variables, bottom=couts_fixes, color='#d62728', label='Coût Variable (Temps)')

plt.title('Impact de l\'optimisation temporelle sur le coût total\n(Pour un taux de satisfaction de 100%)', pad=15)
plt.ylabel('Coût total (€)')
plt.legend()

# Ajouter les valeurs sur les barres
for i in range(len(modeles)):
    plt.text(i, couts_fixes[i]/2, f"{couts_fixes[i]}€", ha='center', va='center', color='white', fontweight='bold')
    plt.text(i, couts_fixes[i] + couts_variables[i]/2, f"{couts_variables[i]}€", ha='center', va='center', color='white', fontweight='bold')
    total = couts_fixes[i] + couts_variables[i]
    plt.text(i, total + 200, f"Total: {total}€", ha='center', fontweight='bold')

plt.ylim(0, max(couts_fixes + couts_variables) * 1.15)
plt.show()